In [ ]:
import numpy as np
from scipy.signal import resample_poly, kaiserord, firwin, freqz
from obspy import read, Stream, Trace, UTCDateTime

from das_tools.utils import proj_path

In [ ]:
SRC_ROOT = proj_path.data_dir("seismometer")  # X1.N?.xml
SRC_DIR = SRC_ROOT                              # N?/{GP1,GP2,GPZ}.D/
OUT_DIR = proj_path.data_dir("seismometer_decimated")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NET = "X1"
YEAR = 2025
STATIONS = tuple(sorted(d.name for d in SRC_DIR.glob("N*") if d.is_dir()))  # e.g. ("N2",)
CHANNELS = ("GP1", "GP2", "GPZ")  # native sensor axes, not rotated
RECOMPUTE = False  # stations whose output file exists are skipped unless True

FS_RAW = 1000.0
FS_MID = 100.0
FS_OUT = 10.0
F_PASS = 4.0     # lowpass passband edge (Hz)
F_STOP = 5.0     # stopband edge = output Nyquist (Hz)
ATTEN_DB = 80.0  # stopband attenuation of the final lowpass

PAD_S = 60.0   # read this much of the neighbouring day files so the filters run across midnight
EDGE_S = 10.0  # dropped at each real data edge (gaps, start/end) for filter transients

### Filters

The final-stage lowpass is designed at `FS_MID`. `resample_poly` applies it polyphase with its group delay removed, so output sample $k$ sits exactly at input time $t_0 + k/f_{s,\mathrm{out}}$.

In [3]:
Q1 = int(round(FS_RAW / FS_MID))
Q2 = int(round(FS_MID / FS_OUT))

numtaps, beta = kaiserord(ATTEN_DB, (F_STOP - F_PASS) / (0.5 * FS_MID))
numtaps |= 1  # odd length -> integer group delay
H_LP = firwin(numtaps, 0.5 * (F_PASS + F_STOP), window=("kaiser", beta), fs=FS_MID)

# filter half-lengths in seconds; EDGE_S and PAD_S must cover them
half_s = 10 * Q1 / FS_RAW + (numtaps // 2) / FS_MID  # resample_poly default half_len = 10 * down
assert EDGE_S > half_s and PAD_S > half_s, half_s
print(f"stage 2 FIR: {numtaps} taps, half-length {half_s:.2f} s (both stages)")

f, h = freqz(H_LP, worN=8192, fs=FS_MID)

stage 2 FIR: 503 taps, half-length 2.61 s (both stages)


### Decimate

Each channel is processed day by day, keeping only the previous day's tail and the next day file in memory
(about 2 GB peak). Day pieces are cut on the 10 Hz UTC grid (`[day0, day1)`), so they merge back into
continuous traces wherever the raw data were continuous.

In [ ]:
def lowpass_decimate(x):
    # 1000 Hz samples -> 10 Hz, zero phase, flat to F_PASS
    y = resample_poly(np.asarray(x, dtype=np.float64), 1, Q1)
    return resample_poly(y, 1, Q2, window=H_LP)


def read_safe(path):
    try:
        return read(str(path), format="MSEED")
    except Exception as e:
        print(f"    skip {path.name}: {type(e).__name__}: {e}")
        return Stream()


def grid_index(t, t0):
    # first output sample index at or after time t, for samples at t0 + k / FS_OUT
    return int(np.ceil(round((t - t0) * FS_OUT, 6)))


def decimate_channel(sta, cha):
    files = sorted((SRC_DIR / sta / f"{cha}.D").glob(f"{NET}.{sta}..{cha}.D.{YEAR}.*"))
    out = Stream()
    prev = Stream()
    nxt = read_safe(files[0]) if files else Stream()
    for i, f in enumerate(files):
        cur = nxt
        nxt = read_safe(files[i + 1]) if i + 1 < len(files) else Stream()
        day0 = UTCDateTime(year=YEAR, julday=int(f.name.split(".")[-1]))
        day1 = day0 + 86400

        # copy so merge/split never touch cur, which becomes the next day's prev
        st = (prev.slice(day0 - PAD_S, day0) + cur + nxt.slice(day1, day1 + PAD_S)).copy()
        prev = cur.slice(day1 - PAD_S, day1).copy()
        del cur
        st.merge(method=1, fill_value=None)

        for tr in st.split():
            t_lo = max(day0, tr.stats.starttime + EDGE_S)
            t_hi = min(day1, tr.stats.endtime - EDGE_S)
            if t_hi - t_lo < 1.0 / FS_OUT:
                continue
            # start the raw trace on the 10 Hz UTC grid
            t0 = day0 + grid_index(tr.stats.starttime, day0) / FS_OUT
            n0 = int(round((t0 - tr.stats.starttime) * FS_RAW))
            y = lowpass_decimate(tr.data[n0:])
            i_lo, i_hi = grid_index(t_lo, t0), min(grid_index(t_hi, t0), len(y))
            if i_hi <= i_lo:
                continue
            hdr = dict(tr.stats.items())
            hdr.update(sampling_rate=FS_OUT, starttime=t0 + i_lo / FS_OUT)
            for k in ("npts", "endtime", "delta", "mseed", "_format", "processing"):
                hdr.pop(k, None)
            out += Trace(y[i_lo:i_hi].astype(np.float32), header=hdr)
        print(f"    {f.name}: done")

    out.merge(method=1, fill_value=None)
    return out.split()  # gaps stay as separate traces (miniSEED cannot hold masked arrays)


def out_path(sta):
    return OUT_DIR / f"{NET}.{sta}.{FS_OUT:g}Hz.mseed"


In [5]:
for sta in STATIONS:
    path = out_path(sta)
    if path.exists() and not RECOMPUTE:
        print(f"{sta}: {path.name} exists, skipped")
        continue
    st_sta = Stream()
    for cha in CHANNELS:
        print(f"{sta} {cha}")
        st_sta += decimate_channel(sta, cha)
    st_sta.sort()
    st_sta.write(str(path), format="MSEED", encoding="FLOAT32")
    print(f"{sta}: wrote {path}")

N2: X1.N2.10Hz.mseed exists, skipped
